In [ ]:
# --- Quick Evaluation Cell (NO TRAINING, WITH SAVING) ---
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

results = {}
global_y_true = []
global_y_pred = []

# Grab the shape variables needed to build the empty skeleton
N_FEATURES = X_all[1].shape[2]
N_CLASSES  = len(le.classes_)

print("Loading and evaluating 8 saved .pth files...")

for test_subject in range(1, 9):
    # 1. Recreate the EXACT scaler for this fold (no training here, just math scaling)
    x_train = np.vstack([X_all[i] for i in range(1, 9) if i != test_subject])
    n_train, W, F = x_train.shape
    scaler = StandardScaler()
    scaler.fit(x_train.reshape(-1, F))

    # 2. Scale the test data
    x_test = X_all[test_subject]
    y_true = y_all[test_subject]
    
    x_test_scaled = scaler.transform(x_test.reshape(-1, F)).reshape(x_test.shape[0], W, F)
    x_test_t = torch.tensor(x_test_scaled.transpose(0, 2, 1), dtype=torch.float32)
    test_loader = DataLoader(TensorDataset(x_test_t), batch_size=64)

    # 3. Build the empty skeleton and LOAD the saved brain (.pth)
    model = PAMAP2_CNN(N_FEATURES, N_CLASSES).to(device)
    model.load_state_dict(torch.load(f'cnn_fold_{test_subject}.pth', map_location=device, weights_only=True))
    
    # LOCK the model to strictly evaluate (disables dropout, etc.)
    model.eval()

    # 4. Predict instantly
    all_preds = []
    with torch.no_grad():
        for Xb in test_loader:
            all_preds.extend(model(Xb[0].to(device)).argmax(1).cpu().numpy())
    
    # 5. Store the metrics
    acc = accuracy_score(y_true, all_preds)
    mac_f1 = f1_score(y_true, all_preds, average='macro')
    
    results[test_subject] = {'accuracy': acc, 'macro_f1': mac_f1}
    
    global_y_true.extend(y_true)
    global_y_pred.extend(all_preds)
    
    print(f"Subject {test_subject} evaluated. (Acc: {acc:.4f}, F1: {mac_f1:.4f})")

# --- Plotting the Results ---
subjects = list(results.keys())
accs = [results[s]['accuracy'] for s in subjects]
f1s = [results[s]['macro_f1'] for s in subjects]

print("\n--- Generating Final Graphs ---")

# 1. Accuracy Bar Chart
plt.figure(figsize=(8, 6))
plt.bar(subjects, accs, color='steelblue')
plt.axhline(y=np.mean(accs), color='red', linestyle='--', label=f'Mean Acc: {np.mean(accs):.4f}')
plt.xlabel('Subject')
plt.ylabel('Accuracy')
plt.title('LOSO Accuracy per Subject (Saved CNN Models)')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('loso_accuracy_saved_cnn.png', dpi=150)
plt.show()

# 2. F1 Score Bar Chart
plt.figure(figsize=(8, 6))
plt.bar(subjects, f1s, color='mediumpurple')
plt.axhline(y=np.mean(f1s), color='red', linestyle='--', label=f'Mean F1: {np.mean(f1s):.4f}')
plt.xlabel('Subject')
plt.ylabel('Macro F1-Score')
plt.title('LOSO Macro F1-Score per Subject (Saved CNN Models)')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.savefig('loso_f1_saved_cnn.png', dpi=150)
plt.show()

# 3. Global Confusion Matrix
cm = confusion_matrix(global_y_true, global_y_pred)
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_.astype(str))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45) 
plt.title('Global LOSO Confusion Matrix (Saved CNN Models)', fontsize=14)
plt.tight_layout()
plt.savefig('global_confusion_matrix_saved_cnn.png', dpi=150)
plt.show()

ModuleNotFoundError: No module named 'torch'